# Explore taxonomy RDF — local SPARQL queries

Uses `rdflib` to load the generated TTL bundles locally and run SPARQL queries interactively.  
No server required — edit queries in-place and re-run cells.

Once you are happy with a query, copy it to the **SPARQLQueries** repository.

**Bundles loaded:**
- `output/bundles/all_gpml_taxonomy_extra-*.ttl` — species annotations (Viridiplantae + per-node)
- `output/bundles/all-*.ttl` — core WikiPathways RDF (optional, large ~300 MB)

**Prefixes used:**
```
wp:   http://vocabularies.wikipathways.org/wp#
ncbi: http://purl.obolibrary.org/obo/NCBITaxon_
pmw:  http://rdf-plantmetwiki.bioinformatics.nl/
```

In [ ]:
from rdflib import Graph, Namespace
from rdflib.plugins.sparql import prepareQuery
import pandas as pd
from pathlib import Path

# ── Namespaces ────────────────────────────────────────────────────────────────
WP   = Namespace("http://vocabularies.wikipathways.org/wp#")
NCBI = Namespace("http://purl.obolibrary.org/obo/NCBITaxon_")
PMW  = Namespace("http://rdf-plantmetwiki.bioinformatics.nl/")

PREFIXES = """
PREFIX wp:   <http://vocabularies.wikipathways.org/wp#>
PREFIX ncbi: <http://purl.obolibrary.org/obo/NCBITaxon_>
PREFIX pmw:  <http://rdf-plantmetwiki.bioinformatics.nl/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
"""

def sparql(g: Graph, query: str) -> pd.DataFrame:
    """Run a SPARQL SELECT and return a DataFrame."""
    results = g.query(PREFIXES + query)
    return pd.DataFrame(results, columns=[str(v) for v in results.vars])

print("Ready.")

## Load the taxonomy extra bundle

This is the small bundle (~4 MB) — fast to load, contains all `wp:organism` triples.

In [ ]:
VERSION = "plantcyc17.0.0-gpml2021"
taxonomy_file = Path(f"../output/bundles/all_gpml_taxonomy_extra-{VERSION}.ttl")

print(f"Loading {taxonomy_file} ...")
g_tax = Graph()
g_tax.parse(str(taxonomy_file), format="turtle")
print(f"Loaded {len(g_tax):,} triples")

---
## 1. Pathway-level: Viridiplantae coverage

Every pathway and reaction should have `wp:organism ncbi:33090` (Viridiplantae).

In [ ]:
sparql(g_tax, """
SELECT (COUNT(DISTINCT ?pathway) AS ?pathways_with_viridiplantae)
WHERE {
    ?pathway wp:organism ncbi:33090 .
}
""")

In [ ]:
# Show a sample of pathway URIs with Viridiplantae
sparql(g_tax, """
SELECT ?pathway
WHERE {
    ?pathway wp:organism ncbi:33090 .
}
LIMIT 10
""")

---
## 2. DataNode-level: per-species annotations

Genes, proteins, and metabolites with a SPECIES field carry individual `wp:organism` triples.

In [ ]:
# How many DataNodes have a species annotation (excluding Viridiplantae)
sparql(g_tax, """
SELECT (COUNT(DISTINCT ?node) AS ?annotated_nodes)
WHERE {
    ?node wp:organism ?species .
    FILTER(?species != ncbi:33090)
}
""")

In [ ]:
# Species distribution — top 20
sparql(g_tax, """
SELECT ?species (COUNT(DISTINCT ?node) AS ?node_count)
WHERE {
    ?node wp:organism ?species .
    FILTER(?species != ncbi:33090)
}
GROUP BY ?species
ORDER BY DESC(?node_count)
LIMIT 20
""")

In [ ]:
# Sample nodes annotated with Arabidopsis thaliana (ncbi:3702)
sparql(g_tax, """
SELECT ?node
WHERE {
    ?node wp:organism ncbi:3702 .
}
LIMIT 10
""")

---
## 3. Combined: pathways containing a specific species

Find all pathways that have at least one node annotated with a given taxon.

In [ ]:
# Pathways containing at least one Arabidopsis thaliana node (ncbi:3702)
# Relies on the URI structure: pathway URI is a prefix of DataNode URIs
sparql(g_tax, """
SELECT DISTINCT ?pathway (COUNT(DISTINCT ?node) AS ?arabidopsis_nodes)
WHERE {
    ?pathway wp:organism ncbi:33090 .
    ?node wp:organism ncbi:3702 .
    FILTER(STRSTARTS(STR(?node), STR(?pathway)))
}
GROUP BY ?pathway
ORDER BY DESC(?arabidopsis_nodes)
LIMIT 20
""")

---
## 4. Multi-species pathways

Pathways where DataNodes span more than one species.

In [ ]:
sparql(g_tax, """
SELECT ?pathway (COUNT(DISTINCT ?species) AS ?species_count)
WHERE {
    ?pathway wp:organism ncbi:33090 .
    ?node wp:organism ?species .
    FILTER(STRSTARTS(STR(?node), STR(?pathway)))
    FILTER(?species != ncbi:33090)
}
GROUP BY ?pathway
HAVING (?species_count > 1)
ORDER BY DESC(?species_count)
LIMIT 20
""")

---
## 5. Biological entity URIs with species

The taxonomy script also annotates the biological identifier URI (e.g. UniProt, TAIR) with `wp:organism`.

In [ ]:
# UniProt URIs with species annotations
sparql(g_tax, """
SELECT ?entity ?species
WHERE {
    ?entity wp:organism ?species .
    FILTER(STRSTARTS(STR(?entity), "https://identifiers.org/uniprot/"))
    FILTER(?species != ncbi:33090)
}
LIMIT 10
""")

In [ ]:
# TAIR gene name URIs with species
sparql(g_tax, """
SELECT ?entity ?species
WHERE {
    ?entity wp:organism ?species .
    FILTER(STRSTARTS(STR(?entity), "https://identifiers.org/tair.name/"))
}
LIMIT 10
""")

---
## 6. Sandbox — write your own queries here

Edit the query below and re-run. When it looks good, copy it to the **SPARQLQueries** repository.

In [ ]:
sparql(g_tax, """
SELECT *
WHERE {
    # ← write your query here
}
LIMIT 20
""")

---
## Optional: load the core bundle

The core bundle is ~300 MB and takes a minute to load. Only needed if you want to query DataNode types, labels, Xrefs, etc. alongside the taxonomy triples.

In [ ]:
core_file = Path(f"../output/bundles/all-{VERSION}.ttl")

print(f"Loading {core_file} ({core_file.stat().st_size / 1e6:.0f} MB) ...")
g_core = Graph()
g_core.parse(str(core_file), format="turtle")
print(f"Loaded {len(g_core):,} triples")

In [ ]:
# Merge taxonomy + core into one graph for cross-layer queries
g_all = g_core + g_tax
print(f"Combined graph: {len(g_all):,} triples")

In [ ]:
# Example cross-layer query: node label + species
sparql(g_all, """
SELECT ?node ?label ?species
WHERE {
    ?node wp:organism ?species .
    ?node rdfs:label  ?label .
    FILTER(?species != ncbi:33090)
}
LIMIT 20
""")